# Conversational recipe agent
Project 25: *Conversational Recommendation Agent with Memory and Tool Use* 

This notebook walks through the project with the code in `src/`: one section per project specification, then the optional extensions, with the main results at each step.

**How to run.** From the repository root: `pip install -r requirements.txt notebook`, then `jupyter notebook notebooks/recipe_agent.ipynb` and *Run All*.

 Live runs use the rule-based extractor; results of the LLM extractor (gpt-oss-120b) are read from the stored result files in `results/`, and every
model output is also kept turn by turn in `logs/`.

The summary tables can also be printed from the command line with `python -m src.report`.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)

from src import report as rt
from src.tools import RecipeDB, search_recipes, get_recipe_details, validate_constraints, diagnose_constraints
from src.schema import Constraint, Objective
from src.state import DialogueState
from src.extract_rules import RuleExtractor
from src.scenarios import load_scenarios, validate
from src.schema import TurnExtraction
from src.agent import Agent, verify_claims
from src.run_agent import run

db = RecipeDB.load()
rules = RuleExtractor()
table = lambda rows: pd.DataFrame(rows).round(3).fillna("-")
print(f"{len(db):,} recipes loaded")

5,000 recipes loaded


## Specification 1: Recipe exploration and normalisation
`src/data_prep.py` builds the database from the raw HUMMUS file (the per-serving checks are recorded in `docs/data_notes.md`). 

Below is the presentation of the filtering steps, the derived dietary labels, and the distributions of preparation time and calories.

In [2]:
display(table(rt.filtering()))
recipes = list(db.recipes.values())
labels = ["vegetarian", "vegan", "gluten_free", "dairy_free", "nut_free"]
print({d: f"{sum(r[d] for r in recipes) / len(recipes):.1%}" for d in labels})
time, kcal = rt.distributions()
display(table(time).set_index("minutes").T, table(kcal).set_index("kcal_per_serving").T)

,step,recipes
0,Raw HUMMUS recipe file,507335
1,"After duration, tag, nutrition and rating filters",87141
2,After ingredient parsing (>= 3 lines),86368
3,Stratified sample (60% main dishes),5000


{'vegetarian': '39.6%', 'vegan': '10.3%', 'gluten_free': '53.5%', 'dairy_free': '39.8%', 'nut_free': '89.8%'}


minutes,<=15,15-30,30-45,45-60,60-90,90-120,120-180,>180
recipes,624,1280,963,710,655,180,181,407


kcal_per_serving,<200,200-400,400-600,600-800,800-1000,>=1000
recipes,1178,1671,1147,512,243,249


## Specification 2: Dialogue dataset
Each scenario is a scripted dialogue whose every turn is annotated with the constraint updates, the expected active constraints afterwards (with origin turn and persistent flag) and the expected action. The validator replays the annotations through the memory rules and checks feasibility against the database.

In [3]:
for split in ("dev", "test", "test2", "long"):
    problems = sum(len(validate(s, db)) for s in load_scenarios(split))
    print(f"{split:6} {len(load_scenarios(split)):3} scenarios, {problems} validation problems")
display(table(rt.scenario_stats()))

dev_001 = load_scenarios("dev")[0]
for i, t in enumerate(dev_001["turns"], 1):
    active = "; ".join(Constraint.from_dict(a).describe() + f" (turn {a['source_turn']})" for a in t["active"])
    print(f"{i}. {t['user']:40} -> {t['expected_action']:9} | {active}")

dev     30 scenarios, 0 validation problems


test    70 scenarios, 0 validation problems


test2   60 scenarios, 0 validation problems


long    10 scenarios, 0 validation problems


,set,scenarios,user_turns,turns_per_scenario,recommend,clarify,report_infeasible,competing_objectives,resets,persistent_updates,removals
0,Dev,30,108,3-4,97,4,7,3,5,11,9
1,Held-out,70,267,3-5,236,12,19,11,11,24,21
2,Held-out 2,60,226,3-5,200,8,18,9,9,22,20
3,Long,10,101,9-12,97,2,2,4,8,8,7


1. Suggest a vegetarian dinner.             -> recommend | vegetarian (turn 1); course: main-dish (turn 1)
2. It should take under 25 minutes.         -> recommend | vegetarian (turn 1); course: main-dish (turn 1); time at most 25 min (turn 2)
3. And please avoid mushrooms.              -> recommend | vegetarian (turn 1); course: main-dish (turn 1); time at most 25 min (turn 2); no mushroom (turn 3)
4. Actually, up to 30 minutes is fine.      -> recommend | vegetarian (turn 1); course: main-dish (turn 1); time at most 30 min (turn 4); no mushroom (turn 3)


## Specification 3: Constraint extraction, rules vs. LLM
Both extractors map a user turn to the same schema. The rule-based extractor runs live here, the LLM extractor's scores come from the stored evaluation.

In [4]:
for utterance in ["Suggest a vegetarian dinner.", "Under 25 minutes, and avoid mushrooms.",
                  "Actually, up to 30 minutes is fine.", "I'm allergic to nuts.",
                  "Which one is the quickest and which has the most protein?"]:
    print(f"{utterance:60} -> {rules.extract(utterance).to_dict()}")
display(table(rt.extraction()), table(rt.extraction_errors()))

Suggest a vegetarian dinner.                                 -> {'updates': [{'op': 'add', 'constraint': {'kind': 'diet', 'target': 'vegetarian'}, 'persistent': False}, {'op': 'add', 'constraint': {'kind': 'course', 'target': 'main-dish'}, 'persistent': False}], 'objectives': [], 'reset': False}
Under 25 minutes, and avoid mushrooms.                       -> {'updates': [{'op': 'add', 'constraint': {'kind': 'max', 'target': 'duration_min', 'value': 25.0}, 'persistent': False}, {'op': 'add', 'constraint': {'kind': 'exclude', 'target': 'mushroom'}, 'persistent': False}], 'objectives': [], 'reset': False}
Actually, up to 30 minutes is fine.                          -> {'updates': [{'op': 'add', 'constraint': {'kind': 'max', 'target': 'duration_min', 'value': 30.0}, 'persistent': False}], 'objectives': [], 'reset': False}
I'm allergic to nuts.                                        -> {'updates': [{'op': 'add', 'constraint': {'kind': 'diet', 'target': 'nut_free'}, 'persistent': True}], 'ob

,set,extractor,P,R,F1,state_exact,state_exact_no_memory,provenance_acc
0,Dev,Rules,1.000,1.000,1.000,1.000,0.315,0.990
1,Dev,LLM,1.000,1.000,1.000,1.000,-,0.997
2,Held-out,Rules,0.967,0.907,0.936,0.783,0.273,0.968
3,Held-out,LLM,0.969,0.966,0.968,0.925,-,0.987
4,Held-out 2,Rules,0.972,0.895,0.932,0.721,0.274,0.975
5,Held-out 2,LLM,0.968,0.971,0.970,0.903,0.288,0.976
6,Long,Rules,0.991,0.933,0.961,0.752,0.109,0.995
7,Long,LLM,1.000,0.992,0.996,0.970,0.119,1.000


,error_type,rules,llm
0,Objectives,10,0
1,Courses,7,0
2,Resets,6,4
3,Inclusions,6,1
4,Exclusions,4,5
5,Diets,4,5
6,Numeric limits,4,4
7,Removals,3,4
8,Total,44,23


## Specification 4: Dialogue state and memory
The state keeps every constraint with the turn it came from and a persistent flag. A new numeric value replaces the old one, diets and ingredients accumulate, and a reset keeps persistent preferences. Without memory the state is rebuilt from each turn alone; the table below shows how often the state is exactly right at each turn of the long dialogues.

In [5]:
state = DialogueState()
for i, t in enumerate(dev_001["turns"], 1):
    state.apply(rules.extract(t["user"], state), i)
    print(f"turn {i}: {t['user']:38} state: {state.describe()}")
key = lambda active: sorted((a["kind"], a["target"], a.get("value"), a["source_turn"], a["persistent"]) for a in active)
print("state, origin turns and persistent flags match the annotation:", key(state.to_active()) == key(dev_001["turns"][-1]["active"]))
display(table(rt.state_by_turn()))

turn 1: Suggest a vegetarian dinner.           state: vegetarian (turn 1); course: main-dish (turn 1)
turn 2: It should take under 25 minutes.       state: vegetarian (turn 1); course: main-dish (turn 1); time at most 25 min (turn 2)
turn 3: And please avoid mushrooms.            state: vegetarian (turn 1); course: main-dish (turn 1); time at most 25 min (turn 2); no mushroom (turn 3)
turn 4: Actually, up to 30 minutes is fine.    state: vegetarian (turn 1); course: main-dish (turn 1); no mushroom (turn 3); time at most 30 min (turn 4)
state, origin turns and persistent flags match the annotation: True


,turn,dialogues,llm,rules,llm_memory_off
0,1,10,1.0,0.9,1.0
1,2,10,1.0,0.9,0.0
2,3,10,1.0,0.9,0.0
3,4,10,1.0,0.9,0.0
4,5,10,1.0,0.8,0.0
5,6,10,1.0,0.7,0.0
6,7,10,1.0,0.6,0.0
7,8,10,1.0,0.5,0.0
8,9,10,0.9,0.6,0.1


## Specification 5: Retrieval and validation tools
Four deterministic tools; no model judges a constraint.

In [6]:
wanted = [Constraint("diet", "vegetarian"), Constraint("course", "main-dish"),
          Constraint("max", "duration_min", 25), Constraint("exclude", "mushroom")]
found = search_recipes(db, constraints=wanted, k=3)
print(found["total_matches"], "matching recipes; top 3:", [r["title"] for r in found["results"]])
check = validate_constraints(db, found["results"][0]["recipe_id"], wanted)
for c in check["checks"]:
    print(f"  {c['description']:28} {'pass' if c['satisfied'] else 'FAIL'}  ({c['evidence']})")
impossible = [Constraint("diet", "vegan"), Constraint("course", "breakfast"),
              Constraint("min", "protein_g", 25), Constraint("max", "duration_min", 15)]
print(diagnose_constraints(db, impossible))
display(table(rt.tool_usage()))

116 matching recipes; top 3: ['Quinoa Black Bean Salad', 'Easy Crescent Samosa (Indian Style Sandwiches)', 'Vegetarian Tacos']
  vegetarian                   pass  (vegetarian: no conflicting ingredients)
  course: main-dish            pass  (courses: main-dish, salads)
  time at most 25 min          pass  (duration_min = 15 min (<= 25))
  no mushroom                  pass  (no ingredient matching 'mushroom')
{'total_matches': 0, 'per_constraint': [{'constraint': {'kind': 'diet', 'target': 'vegan'}, 'description': 'vegan', 'matches_if_dropped': 5}, {'constraint': {'kind': 'course', 'target': 'breakfast'}, 'description': 'course: breakfast', 'matches_if_dropped': 5}, {'constraint': {'kind': 'min', 'target': 'protein_g', 'value': 25.0}, 'description': 'protein at least 25 g', 'matches_if_dropped': 9, 'closest_available': 9.4}, {'constraint': {'kind': 'max', 'target': 'duration_min', 'value': 15.0}, 'description': 'time at most 15 min', 'matches_if_dropped': 0}]}


,tool,calls
0,diagnose_constraints,21.0
1,get_recipe_details,687.0
2,search_recipes,288.0
3,validate_constraints,687.0
4,calls per turn,6.3
5,candidates rejected by validation,0.0


## Specifications 6 and 7: Decision policy and grounded explanations
The agent clarifies vague requests, reports infeasible ones with a diagnosis instead of relaxing a constraint, and shows trade-offs between competing objectives. 

In [7]:
def chat(*turns, **options):
    agent = Agent(db, rules, **options)
    for user in turns:
        rec = agent.turn(user)
        unsupported = len(verify_claims(db, rec["claims"]))
        print(f"USER:  {user}\nAGENT [{rec['action']}, {len(rec['claims'])} claims, {unsupported} unsupported]: {rec['message']}\n")

chat("Suggest a vegetarian dinner.", "Under 25 minutes.", "Avoid mushrooms.", "Actually, up to 30 minutes is fine.")

USER:  Suggest a vegetarian dinner.
AGENT [recommend, 15 claims, 0 unsupported]: I recommend Easy Peezy Pizza Dough (Bread Machine Pizza Dough) (195 min, 1727.2 kcal, 42 g protein per serving). It meets all your requirements (vegetarian; course: main-dish): vegetarian: no conflicting ingredients; courses: main-dish, lunch, breads. Also verified: Baked Stuffed Pasta Shells (55 min, 397.5 kcal, 23.2 g protein per serving); Quinoa Black Bean Salad (15 min, 190.1 kcal, 10 g protein per serving).

USER:  Under 25 minutes.
AGENT [recommend, 18 claims, 0 unsupported]: I recommend Quinoa Black Bean Salad (15 min, 190.1 kcal, 10 g protein per serving). It meets all your requirements (vegetarian; course: main-dish; time at most 25 min): vegetarian: no conflicting ingredients; courses: main-dish, salads; duration_min = 15 min (<= 25). Also verified: Easy Crescent Samosa (Indian Style Sandwiches) (25 min, 163.8 kcal, 4.7 g protein per serving); Vegetarian Tacos (20 min, 334.5 kcal, 9.4 g protein p

In [8]:
chat("Something healthy, please.",                                   # vague -> clarify
     "I'm vegan.",                                                    # persistent preference
     "A breakfast with at least 25 g of protein in under 15 minutes.",  # infeasible -> diagnosis
     "OK, 5 g of protein is enough.")                                 # relaxed by the user

USER:  Something healthy, please.
AGENT [clarify, 0 claims, 0 unsupported]: Happy to help. To narrow it down: what kind of meal are you after (breakfast, lunch, dinner, dessert, ...), do you have any dietary restrictions or ingredients to avoid, and how much time do you have?

USER:  I'm vegan.
AGENT [recommend, 12 claims, 0 unsupported]: I recommend Easy Peezy Pizza Dough (Bread Machine Pizza Dough) (195 min, 1727.2 kcal, 42 g protein per serving). It meets all your requirements (vegan): vegan: no conflicting ingredients. Also verified: Pan Release, Professional Pan Coating (Better Than Pam Spray!) (5 min, 1398.1 kcal, 4.3 g protein per serving); Frito Lay Bean Dip (15 min, 100.2 kcal, 5.8 g protein per serving).

USER:  A breakfast with at least 25 g of protein in under 15 minutes.
AGENT [report_infeasible, 2 claims, 0 unsupported]: I couldn't find any recipe in the database that is vegan; course: breakfast; time at most 15 min; protein at least 25 g. Without the requirement "protein

In [9]:
chat("A main dish under 350 calories.", "Which is faster and which has more protein?")   # trade-off
display(table(rt.policy()))

USER:  A main dish under 350 calories.
AGENT [recommend, 15 claims, 0 unsupported]: I recommend Tortellini Tomato Spinach Soup (30 min, 312.6 kcal, 17.1 g protein per serving). It meets all your requirements (course: main-dish; calories at most 350 kcal): courses: main-dish, soups-stews; calories_kcal = 312.6 kcal per serving (<= 350). Also verified: Crispy Baked Chicken Leg Quarters (Very Easy, One Dish) (90 min, 336.5 kcal, 34.2 g protein per serving); Armenian Herb Marinade Grilled Chicken Breasts (30 min, 276.5 kcal, 18.1 g protein per serving).

USER:  Which is faster and which has more protein?
AGENT [recommend, 10 claims, 0 unsupported]: These objectives pull in different directions, so here is the trade-off: the quickest option is Simple Savory Pasta (3 min, 50.2 kcal, 0.6 g protein per serving); the highest protein option is Churros de Pescado (Fried Fish Fingers) (14 min, 328.9 kcal, 54.1 g protein per serving). Both meet all your requirements (course: main-dish; calories at 

,set,extractor,action_acc,clarify_recall,infeasible_recall,relaxations_state_wrong,relaxations_state_correct,wrong_actions_state_wrong,wrong_actions_state_correct
0,Held-out,Rules,0.951,0.750,0.895,29,0,11,2
1,Held-out,LLM,0.974,0.750,0.947,3,0,5,2
2,Held-out 2,Rules,0.942,0.875,0.889,21,0,12,1
3,Held-out 2,LLM,0.974,0.875,0.944,3,0,5,1


## Specification 8: Evaluation and ablation
Five configurations: the conversational RAG baseline, RAG with structured memory, the full agent, and the full agent without memory or without validation. First a live re-run of the full agent with the rule-based extractor on the 70 held-out dialogues, compared with the stored result; then the comparison on both held-out sets and the claim totals.

In [10]:
live, _ = run(Agent(db, rules), load_scenarios("test"), db)
stored = rt.load("test_agent")
for key in ("constraint_satisfaction", "silent_relaxations", "infeasible_recall", "action_accuracy"):
    print(f"{key:24} live {live[key]}   stored {stored[key]}")
display(table(rt.systems()), table(rt.claims()))

constraint_satisfaction  live 0.8934   stored 0.8934
silent_relaxations       live 29   stored 29
infeasible_recall        live 0.8947   stored 0.8947
action_accuracy          live 0.9513   stored 0.9513


,system,test_rules_satisfaction,test_rules_relaxations,test_llm_satisfaction,test_llm_relaxations,test2_rules_satisfaction,test2_rules_relaxations,test2_llm_satisfaction,test2_llm_relaxations
0,RAG baseline,0.248,237,0.248,237,0.304,196,0.304,196
1,RAG + memory,0.834,50,0.911,24,0.830,44,0.893,24
2,Full agent,0.893,29,0.990,3,0.920,21,0.988,3
3,"Agent, no memory",0.406,141,0.434,137,0.386,125,0.402,125
4,"Agent, no validation",0.259,224,0.257,227,0.272,194,0.283,193


,set,runs,claims,unsupported
0,Held-out,10,27491,0
1,Held-out 2,9,21176,0
2,Long,9,10426,0


## Optional extensions
Preference-based ranking reorders only the recipes that satisfy every constraint.

The reviewer checks explanations written by the LLM against the tool output (stored results). 

Longer conversations are covered in Specification 4 above (state accuracy by turn on the long dialogues).

In [11]:
for ranking in ("popularity", "preference"):
    agent = Agent(db, rules, ranking=ranking)
    rec = agent.turn("Can you recommend a vegetarian main course?")
    print(f"{ranking:10}: " + "; ".join(r["title"] for r in rec["recommendations"]))
display(table(rt.preference_ranking()), table(rt.reviewer()))

popularity: Easy Peezy Pizza Dough (Bread Machine Pizza Dough); Baked Stuffed Pasta Shells; Quinoa Black Bean Salad
preference: Baked Stuffed Pasta Shells; 3 Cheese Manicotti; Easy Veggie Enchiladas With Quick Sauce


,extractor,ranking,satisfaction,relaxations,mean_rating,mean_kcal,mean_minutes,share_100_800_kcal
0,Rules,popularity,0.893,29,4.73,395,47.6,0.80
1,Rules,preference,0.896,29,4.73,401,34.2,0.95
2,LLM,popularity,0.990,3,4.72,390,49.8,0.81
3,LLM,preference,0.987,3,4.73,402,36.7,0.95


,set,turns_reviewed,claims_checked,unsupported,turns_corrected
0,Dev,104,630,0,0
1,Held-out,258,1575,0,0


## Reproducing everything
| What | Command |
|---|---|
| Summary tables (also saved as CSV in `results/summary/`) | `python -m src.report` |
| Validate the scenarios | `python -m src.scenarios test` (also `dev`, `test2`, `long`) |
| Extraction and state accuracy | `python -m src.evaluate --split test --extractor rules [--no-memory]` |
| A system configuration | `python -m src.run_agent --split test --mode agent [--no-memory \| --no-validate]` |
| With the LLM extractor (needs an API key in `.env`) | add `--extractor llm --provider lehmus` |
| Unit tests | `python -m pytest -q` |